# Space Syntax — Gallery Floor Plan

This tutorial follows the Spatial Intelligence notebook: load the gallery, build graphs, analyse them, and colour the original geometry.

**Visibility results colour floor cells. Axial and segment results colour lines.** Each floor cell has its own sample node; values transfer directly without interpolation.

Run the cells from top to bottom using the updated TopologicPy checkout from our compatibility work and the PythonOCC backend. The supplied `Supporting Files/gallery.obj` is the same gallery as the original notebook. Radii and grid spacing use the OBJ's model units.

Building-analysis reference: [Space Syntax / DepthmapX tutorial](https://www.spacesyntax.online/software-and-manuals/depthmap/building-spatial-model/). This notebook demonstrates TopologicPy's models; identical DepthmapX values require identical sampling, maps and settings.

## 1. Imports and settings
Start with a coarse grid. Reduce `GRID_SPACING` later to test how resolution affects the results.

In [ ]:
from pathlib import Path
import csv
import math
import time
import plotly.graph_objects as go

from topologicpy.Cluster import Cluster
from topologicpy.Color import Color
from topologicpy.Core import Core
from topologicpy.Dictionary import Dictionary
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Grid import Grid
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly
from topologicpy.TGraph import TGraph
from topologicpy.Topology import Topology
from topologicpy.Vertex import Vertex
from topologicpy.Wire import Wire

OBJ_PATH = Path("gallery.obj")
GRID_SPACING = 4.0
LOCAL_RADIUS = 3                      # visibility/axial graph steps
ANGULAR_RADIUS = 1                    # quarter-turn units (1 = 90 degrees)
METRIC_RADIUS = 30                    # model units along segment routes
SEGMENT_STEP_RADIUS = 3
RENDERER = "plotly_mimetype"           # use "browser" for a separate window
OUTPUT_DIR = Path("results")

print("TopologicPy:", Helper.Version(check=False))
print("Backend:", Core.Backend().__class__.__name__)
assert hasattr(TGraph, "AngularTulipAnalysis") and hasattr(TGraph, "AnalysisContext"), "Use the updated TopologicPy checkout."
assert OBJ_PATH.is_file(), "Keep Supporting Files beside this notebook, or update OBJ_PATH."

## 2. Load the accessible floor
The imported face includes the gallery boundary and internal holes, which block visibility.

In [ ]:
floor_plan = Topology.ByOBJPath(str(OBJ_PATH), removeCoplanarFaces=True, transposeAxes=False)[0]
assert Topology.IsInstance(floor_plan, "Face"), "The gallery should import as one floor face."
print("Floor area:", Face.Area(floor_plan), "square model units")
print("Internal boundaries:", len(Face.InternalBoundaries(floor_plan)))

### Small display helpers
These helpers apply one colour scale and keep each analysis under its own dictionary key. Grey means the measure is undefined. Each map has its own numerical colour range.

In [ ]:
SYNTAX_SCALE = Color.ColorScale("syntax")

def store_values(graph, values, key):
    records = TGraph.Vertices(graph)
    assert values is not None and len(values) == len(records)
    valid = [float(v) for v in values if v is not None and math.isfinite(v) and v >= 0]
    low, high = (min(valid), max(valid)) if valid else (0.0, 1.0)
    for record, value in zip(records, values):
        value = float(value) if value is not None else -1.0
        if not math.isfinite(value):
            value = -1.0
        colour = "#7f7f7f" if value < 0 else Color.AnyToHex(
            Color.ByValueInRange(value, minValue=low, maxValue=high, colorScale=SYNTAX_SCALE))
        TGraph.SetVertexValue(graph, record["index"], key, value)
        TGraph.SetVertexValue(graph, record["index"], key + "_color", colour)
    return values

def show_plan(topologies, title, key=None, faces=True, viewpoint=None):
    items = topologies if isinstance(topologies, list) else [topologies]
    figure = go.Figure()
    for item in items:
        dictionary = Topology.Dictionary(item)
        value = Dictionary.ValueAtKey(dictionary, key) if key else None
        colour = Dictionary.ValueAtKey(dictionary, key + "_color") if key else None
        label = f"{key}: {value}" if key else title
        if faces:
            points = [Vertex.Coordinates(v) for v in Wire.Vertices(Face.ExternalBoundary(item))]
            points.append(points[0])
            figure.add_trace(go.Scatter(
                x=[p[0] for p in points], y=[p[1] for p in points], mode="lines",
                line=dict(width=0.3, color=colour or "#cccccc"), fill="toself",
                fillcolor=colour or "#eeeeee", hoveron="fills", text=label,
                hovertemplate="%{text}<extra></extra>", showlegend=False))
        else:
            points = [Vertex.Coordinates(Edge.StartVertex(item)), Vertex.Coordinates(Edge.EndVertex(item))]
            figure.add_trace(go.Scatter(
                x=[p[0] for p in points], y=[p[1] for p in points], mode="lines",
                line=dict(width=3, color=colour or "#999999"), text=label,
                hovertemplate="%{text}<extra></extra>", showlegend=False))
    # Leave the floor's internal holes white, including cells clipped by obstacles.
    for hole in Face.InternalBoundaries(floor_plan):
        points = [Vertex.Coordinates(v) for v in Wire.Vertices(hole)]
        points.append(points[0])
        figure.add_trace(go.Scatter(x=[p[0] for p in points], y=[p[1] for p in points],
                                   mode="lines", line=dict(width=0), fill="toself",
                                   fillcolor="white", hoverinfo="skip", showlegend=False))
    wall_x, wall_y = [], []
    for edge in Topology.Edges(floor_plan):
        a, b = Vertex.Coordinates(Edge.StartVertex(edge)), Vertex.Coordinates(Edge.EndVertex(edge))
        wall_x.extend([a[0], b[0], None]); wall_y.extend([a[1], b[1], None])
    figure.add_trace(go.Scatter(x=wall_x, y=wall_y, mode="lines",
                               line=dict(color="#555555", width=1), hoverinfo="skip", showlegend=False))
    if key:
        values = [Dictionary.ValueAtKey(Topology.Dictionary(item), key) for item in items]
        valid = [float(v) for v in values if v is not None and math.isfinite(v) and v >= 0]
        if valid:
            figure.add_trace(go.Scatter(x=[None], y=[None], mode="markers", showlegend=False,
                marker=dict(color=[min(valid)], cmin=min(valid), cmax=max(valid),
                            colorscale=SYNTAX_SCALE, showscale=True,
                            colorbar=dict(title=key.replace("_", " "), thickness=14))))
    if viewpoint is not None:
        x,y,z = Vertex.Coordinates(viewpoint)
        figure.add_trace(go.Scatter(x=[x], y=[y], mode="markers", showlegend=False,
                                   marker=dict(size=9, color="#222222", line=dict(color="white", width=1)),
                                   hovertemplate="Viewpoint<extra></extra>"))
    figure.update_layout(title=title, width=900, height=650, plot_bgcolor="white",
                         paper_bgcolor="white", margin=dict(l=10, r=10, t=50, b=10),
                         xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1))
    figure.show(renderer=RENDERER)
    return figure
def show_result(graph, values, key, title, faces=True):
    store_values(graph, values, key)
    geometry = TGraph.TransferDictionariesToOriginators(graph, keys=[key, key + "_color"])
    assert geometry is not None and len(geometry) == len(TGraph.Vertices(graph))
    assert all(Topology.IsInstance(item, "Face" if faces else "Edge") for item in geometry)
    return show_plan(geometry, title, key=key, faces=faces)

_ = show_plan(floor_plan, "Gallery floor plan")

## 3. Grid, floor cells and interior samples
`Grid.OnFace` creates the cutting lines. Slice the floor and give every cell a stable `cell_id`. Each sample lies inside its own cell; partial cells along walls are retained.

In [ ]:
grid = Grid.OnFace(floor_plan, spacing=GRID_SPACING)
assert grid is not None
sliced_floor = Topology.Slice(floor_plan, grid)
floor_cells = Topology.Faces(sliced_floor)
assert floor_cells

samples = []
samples_by_id = {}
for i, face in enumerate(floor_cells):
    cell_id = f"cell_{i:04d}"
    face = Topology.SetDictionary(face, Dictionary.ByKeyValue("cell_id", cell_id))
    floor_cells[i] = face
    sample = Face.InternalVertex(face)
    assert sample is not None
    sample = Topology.SetDictionary(sample, Dictionary.ByKeyValue("cell_id", cell_id))
    samples.append(sample)
    samples_by_id[cell_id] = sample

print("Floor cells / sample points:", len(floor_cells))
_ = show_plan(floor_cells, "Gallery sliced into analysis cells")

## 4. Build the visibility graph
Join pairs of samples that can see one another within the floor. Use the full visibility graph: no neighbour limit, distance cutoff or added boundary nodes. Link each graph node to its source face by `cell_id`.

In [ ]:
start = time.perf_counter()
visibility_graph = TGraph.VisibilityGraph(
    floor_plan, vertices=samples, includeBoundaryVertices=False,
    maxDistance=None, maxNeighbors=None, trim=True,
    storeEdgeRepresentations=False, ontology=False)
assert visibility_graph is not None

cells_by_id = {Dictionary.ValueAtKey(Topology.Dictionary(face), "cell_id"): face for face in floor_cells}
node_ids = [record["dictionary"]["cell_id"] for record in TGraph.Vertices(visibility_graph)]
assert len(node_ids) == len(set(node_ids)) == len(floor_cells)
assert set(node_ids) == set(cells_by_id)
for record in TGraph.Vertices(visibility_graph):
    TGraph.SetVertexOriginator(visibility_graph, record["index"], cells_by_id[record["dictionary"]["cell_id"]])

visibility_context = TGraph.AnalysisContext(visibility_graph)
print("Visibility nodes:", len(node_ids), "links:", len(TGraph.Edges(visibility_graph)))
print("Graph creation:", round(time.perf_counter() - start, 2), "seconds")
print("Connected components:", len(TGraph.ConnectedComponents(visibility_graph)))

## 5. Visual connectivity
Connectivity is unnormalised graph degree. How many other sample locations can this cell see? Higher values indicate more direct visual connections.

In [ ]:
# Compile once: avoids per-node cache validation on dense visibility graphs.
compiled = TGraph.Compile(visibility_graph)
visual_connectivity = [
    float(compiled["degree_all"][compiled["position"][record["index"]]])
    for record in TGraph.Vertices(visibility_graph)
]
_ = show_result(visibility_graph, visual_connectivity, "visual_connectivity", "Visual connectivity")

## 6. Global visual integration
Hillier–Hanson (HH) integration measures accessibility through the visibility graph. The radius is global; grey cells have undefined HH integration.

In [ ]:
visual_integration = TGraph.Integration(visibility_graph, method="hh", radius=None, key=None, colorKey=None)
_ = show_result(visibility_graph, visual_integration, "visual_integration", "Global visual integration [HH]")

## 7. Local visual integration
Limit analysis to locations within `LOCAL_RADIUS` visibility-graph steps. This is a topological radius, not a distance in metres.

In [ ]:
visual_local = TGraph.Integration(visibility_graph, method="hh", radius=LOCAL_RADIUS, key=None, colorKey=None)
_ = show_result(visibility_graph, visual_local, "visual_integration_local", f"Local visual integration [HH], R{LOCAL_RADIUS}")

## 8. Visual mean depth
The mean number of visibility steps to other reachable sample locations. Lower values indicate shallower visual access; unreachable locations are excluded.

In [ ]:
visual_mean_depth = []
for record in TGraph.Vertices(visibility_graph):
    depths = TGraph.DepthMap(visibility_graph, source=record["index"], key="temporary_depth")
    mean = sum(depths.values()) / (len(depths) - 1) if len(depths) > 1 else -1
    visual_mean_depth.append(mean)
_ = show_result(visibility_graph, visual_mean_depth, "visual_mean_depth", "Visual mean depth — lower is shallower")

## 9. Visual closeness and betweenness
Ordinary graph closeness summarises access to other samples. Betweenness identifies cells lying on shortest visibility-step routes; it is not angular segment Choice.

In [ ]:
visual_closeness = TGraph.ClosenessCentrality(visibility_graph, normalize=False, key=None, colorKey=None)
_ = show_result(visibility_graph, visual_closeness, "visual_closeness", "Visual graph closeness")
visual_betweenness = TGraph.BetweennessCentrality(visibility_graph, normalize=False, key=None, colorKey=None)
_ = show_result(visibility_graph, visual_betweenness, "visual_betweenness", "Visual graph betweenness")

## 10. Create an axial map and segment graph
Axial lines represent long straight sight/movement lines. Segments split those lines at intersections. TopologicPy generates an **approximate sampled axial map**; it is not DepthmapX’s all-lines/fewest-lines generator. Modest sampling limits keep this example small.

In [ ]:
axial_report = Face.AxialEdges(
    floor_plan, reduce=True, samplingDistance=GRID_SPACING,
    maxSamples=300, maxCandidates=1200, returnReport=True)
assert axial_report is not None and axial_report["edges"]
axial_edges = axial_report["edges"]
axial_graph = TGraph.AxialGraph(axial_edges)
segment_graph = TGraph.SegmentGraph(axial_edges)
assert axial_graph is not None and segment_graph is not None
axial_context = TGraph.AnalysisContext(axial_graph)
segment_context = TGraph.AnalysisContext(segment_graph)
segment_edges = TGraph.Originators(segment_graph)

print("Axial lines:", len(axial_edges), "segments:", len(TGraph.Vertices(segment_graph)))
print("Sampled coverage complete:", axial_report["coverageComplete"], "connected:", axial_report["connected"])
print("Sampling limits reached:", axial_report["limitsReached"])
_ = show_plan(axial_edges, "Approximate axial map", faces=False)
_ = show_plan(segment_edges, "Axial lines split into segments", faces=False)

## 11. Axial and segment connectivity
Count connections at line intersections. Angular connectivity also sums turn costs in quarter-turn units. These maps colour lines rather than floor cells.

In [ ]:
axial_connectivity = TGraph.Connectivity(axial_graph, key=None)
_ = show_result(axial_graph, axial_connectivity, "axial_connectivity", "Axial connectivity", faces=False)
segment_connectivity = TGraph.AngularConnectivity(segment_graph, method="degree", key=None)
_ = show_result(segment_graph, segment_connectivity, "segment_connectivity", "Segment connectivity", faces=False)
angular_connectivity = TGraph.AngularConnectivity(segment_graph, method="depthmap", key=None)
_ = show_result(segment_graph, angular_connectivity, "angular_connectivity", "Angular connectivity — sum of turn costs", faces=False)

## 12. Axial integration and Choice
Topological HH integration on the axial graph. The local radius counts line-to-line transitions. Axial Choice counts shortest-step routes passing through each line.

In [ ]:
axial_global = TGraph.Integration(axial_graph, method="hh", key=None, colorKey=None)
_ = show_result(axial_graph, axial_global, "axial_integration", "Global axial integration [HH]", faces=False)
axial_local = TGraph.Integration(axial_graph, method="hh", radius=LOCAL_RADIUS, key=None, colorKey=None)
_ = show_result(axial_graph, axial_local, "axial_integration_local", f"Local axial integration [HH], R{LOCAL_RADIUS}", faces=False)
axial_choice = TGraph.Choice(axial_graph, normalize=False, key=None)
_ = show_result(axial_graph, axial_choice, "axial_choice", "Axial Choice — shortest-step betweenness", faces=False)

## 13. Global angular integration and Choice
Tulip analysis uses angular cost and DepthmapX-style quantisation (`T1024`). Integration describes angular accessibility; Choice describes through-route potential. One combined calculation returns all four angular measures.

In [ ]:
angular_global = TGraph.AngularTulipAnalysis(segment_graph, tulipBins=1024, mantissa=None, writeValues=False)
assert angular_global is not None
_ = show_result(segment_graph, angular_global["integration"], "angular_integration", "Global angular integration, T1024", faces=False)
_ = show_result(segment_graph, angular_global["choice"], "angular_choice", "Global angular Choice, T1024", faces=False)

## 14. Angular, metric and topological radii
The optimisation still uses angular cost. A radius limits destinations by angular turns, midpoint route distance or segment steps. Compare local Integration and Choice for each radius type.

In [ ]:
local_angular_results = {}
for radius_type, radius in [("angular", ANGULAR_RADIUS), ("metric", METRIC_RADIUS), ("topological", SEGMENT_STEP_RADIUS)]:
    result = TGraph.AngularTulipAnalysis(segment_graph, radius=radius, radiusType=radius_type,
                                        tulipBins=1024, mantissa=None, writeValues=False)
    assert result is not None
    local_angular_results[radius_type] = result
    for measure in ["integration", "choice"]:
        key = f"angular_{measure}_{radius_type}"
        _ = show_result(segment_graph, result[measure], key,
                        f"Angular {measure.title()}, {radius_type} R{radius}", faces=False)

## 15. NAIN and NACH
Display normalised angular Integration (NAIN) and Choice (NACH) from the global calculation. These are segment measures, so retain line-based displays.

In [ ]:
_ = show_result(segment_graph, angular_global["nain"], "nain", "Global NAIN", faces=False)
_ = show_result(segment_graph, angular_global["nach"], "nach", "Global NACH", faces=False)

## 16. Segment-length weighting
Weight origin–destination demand by segment length. Compare these with the unweighted global maps above. Weighted NAIN/NACH are available as formula extensions; they are not independently verified standard DepthmapX normalised outputs.

In [ ]:
angular_weighted = TGraph.AngularTulipAnalysis(segment_graph, weighting="length", tulipBins=1024,
                                               mantissa=None, writeValues=False)
assert angular_weighted is not None
_ = show_result(segment_graph, angular_weighted["integration"], "angular_integration_length", "Length-weighted angular integration", faces=False)
_ = show_result(segment_graph, angular_weighted["choice"], "angular_choice_length", "Length-weighted angular Choice", faces=False)

## 17. Isovists at three locations
An isovist is the visible area from one viewpoint. Select the cells with low, middle and high visual connectivity and display each isovist. The dark dot marks the viewpoint.

In [ ]:
ranked = sorted(range(len(visual_connectivity)), key=lambda i: visual_connectivity[i])
selected = [ranked[0], ranked[len(ranked)//2], ranked[-1]]
for label, index in zip(["Low", "Middle", "High"], selected):
    isovist = Face.Isovist(floor_plan, samples_by_id[node_ids[index]], metrics=True)
    assert isovist is not None
    metrics = Topology.Dictionary(isovist)
    print(label, "connectivity viewpoint:", node_ids[index],
          {key: Dictionary.ValueAtKey(metrics, key) for key in ["area", "perimeter", "compactness", "occlusivity"]})
    _ = show_plan(isovist, label + " connectivity viewpoint — isovist", viewpoint=samples_by_id[node_ids[index]])

## 18. Reuse the calculation
The context reuses calculations when the graph and analysis settings remain compatible. Clear it to measure a fresh request, then repeat. Cache retrieval is not another traversal; relevant edits trigger new work.

In [ ]:
segment_context.Clear(resetStatistics=True)
start = time.perf_counter()
fresh = TGraph.AngularTulipAnalysis(segment_graph, mantissa=None, writeValues=False)
fresh_seconds = time.perf_counter() - start
start = time.perf_counter()
reused = TGraph.AngularTulipAnalysis(segment_graph, mantissa=None, writeValues=False)
reused_seconds = time.perf_counter() - start
assert fresh == reused
print("Fresh:", round(fresh_seconds, 4), "s; reused:", round(reused_seconds, 4), "s")
print("Estimated cache storage:", round(segment_context.Info()["estimatedBytes"] / 1024**2, 2), "MiB")

## 19. Export the values
Save separate cell, axial-line and segment tables. IDs and coordinates allow the results to be joined back to their geometry. Each analysis keeps its own result column. Line tables also include their endpoint coordinates.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

def export_nodes(graph, path, identity_key):
    records = TGraph.Vertices(graph)
    rows = []
    for record in records:
        data = record["dictionary"]
        row = {"node_index": record["index"], identity_key: data.get(identity_key, "")}
        row.update(dict(zip(["x", "y", "z"], TGraph.Coordinates(graph, record["index"]))))
        originator = TGraph.VertexOriginator(graph, record["index"])
        if Topology.IsInstance(originator, "Edge"):
            row.update(dict(zip(["x1", "y1", "z1"], Vertex.Coordinates(Edge.StartVertex(originator)))))
            row.update(dict(zip(["x2", "y2", "z2"], Vertex.Coordinates(Edge.EndVertex(originator)))))
        row.update({key: value for key, value in data.items()
                    if isinstance(value, (int, float)) and key not in ["index", "active", "x", "y", "z", "temporary_depth"]})
        rows.append(row)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    print("Saved:", path, "—", len(rows), "rows")

export_nodes(visibility_graph, OUTPUT_DIR / "gallery_visibility_cells.csv", "cell_id")
export_nodes(axial_graph, OUTPUT_DIR / "gallery_axial_lines.csv", "edge_id")
export_nodes(segment_graph, OUTPUT_DIR / "gallery_segments.csv", "edge_id")

## Try one change

Reduce `GRID_SPACING` and rerun the notebook, or change one local radius. Smaller cells provide finer spatial sampling but increase graph construction and analysis time. Boundary cells use interior sample points, so the lattice is not an exact reproduction of DepthmapX's point-grid placement. Keep the same resolution and settings when comparing plans.